In [1]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np
from torch.utils.tensorboard import SummaryWriter

In [2]:
with open('girls-names.csv') as f:
    content = f.readlines()
print(len(content))
content = ' '.join( [c.strip()+'.' for c in content])
content[:100]

# allowed_chars = set(string.ascii_letters + string.digits + ". ")

# with open('persuasion.txt') as f:
#     content = f.read()
# print(len(content))
# content = ''.join([c for c in content if c in allowed_chars])
# content[:100]

3759


'Aadhya. Aadya. Aaira. Aairah. Aaliyah. Aanya. Aaradhya. Aarna. Aarvi. Aarya. Aashvi. Aavya. Aayat. A'

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)

mps


In [4]:
allowed_chars = set(string.ascii_letters + string.digits + ". ")

content = ''.join([c for c in content if c in allowed_chars])

chars = set(sorted(list(content)))
i_to_c = {k:v for k, v in enumerate(chars)}
c_to_i = {k:v for v, k in i_to_c.items()}

encode = lambda s: [c_to_i[c] for c in s] # noqa: E731
decode = lambda s: ''.join([i_to_c[i] for i in s]) # noqa: E731
decode(encode(content[:100]))

encoded_content = torch.tensor(encode(content))

In [5]:
context_length = 4

class NameGenerator(nn.Module):
    def __init__(self, vocab_size, context_length, hidden_dim):
        super().__init__()
        
        input_dim = context_length * vocab_size
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)

        self.net = nn.Sequential(    
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
            )

    def forward(self, x):
        emb = self.embedding(x)
        emb = emb.view(emb.shape[0], -1)
        logits = self.net(emb)
        return logits

model = NameGenerator(len(chars), context_length, 512)
optimizer = torch.optim.Adam(model.parameters(), lr=.01)

In [6]:
from torch.utils.data import TensorDataset, DataLoader

xs = []
ys = []
for i in range(0, len(encoded_content)-context_length):
    x_chunk = encoded_content[i:i+context_length]
    y_chunk = encoded_content[i+context_length]

    xs.append(x_chunk)
    ys.append(y_chunk)

X = torch.stack(xs)
Y = torch.stack(ys)

dataset = TensorDataset(X, Y)

loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [7]:
for xb, yb in loader:
    logits = model.forward(xb)
    
    loss = F.cross_entropy(logits.view(-1, len(chars)), yb)
    print(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

4.002225875854492
3.858088493347168
3.280587673187256
2.4341578483581543
2.61614727973938
2.8917224407196045
2.700251817703247
2.476529121398926
2.639016628265381
2.6513755321502686
2.5678670406341553
2.920464277267456
3.0686888694763184
2.5188565254211426
2.5007808208465576
2.6181318759918213
2.495634078979492
2.6932175159454346
2.863779306411743
2.9759361743927
2.3315184116363525
3.0942628383636475
2.508878469467163
2.4246227741241455
2.3847343921661377
2.2984023094177246
2.265563726425171
2.4164998531341553
2.6361684799194336
2.7603559494018555
2.2588179111480713
2.245176315307617
2.3405704498291016
2.4040205478668213
2.1580257415771484
2.4049153327941895
2.270659923553467
2.832789421081543
2.568369150161743
2.935274839401245
2.3246805667877197
2.3070127964019775
1.699304461479187
2.74657940864563
2.1849093437194824
2.6658525466918945
2.122734308242798
2.536039352416992
2.200003147125244
2.4922969341278076
2.326505661010742
2.363557815551758
2.5336592197418213
2.6337122917175293
1.9

In [8]:
# Girls names
for initial in [ch for ch in string.ascii_letters if ch==ch.upper()]:
    inputs = ' ' * (context_length-1) + initial

    for i in range(15):
        output = decode([model.forward(torch.tensor(encode(inputs[-context_length:])).view(1,-1)).argmax().item()])
        inputs+=output
        if output=='.': break
    print(inputs)


   Ara.
   Bra.
   Cara.
   Dara.
   Ela.
   Fara.
   Gara.
   Hara.
   Ira.
   Ja.
   Kara.
   Lara.
   Mara.
   Nara.
   Ora.
   Para.
   Qa.
   Ranne.
   Sanna.
   Tarna.
   Ura.
   Vara.
   Wara.
   Xara.
   Ya.
   Za.


In [9]:
# Infinite persuasion

# inputs = ' ' * (context_length-1) + 'Z'

# for i in range(1500):
#     output = decode([model.forward(torch.tensor(encode(inputs[-context_length:])).view(1,-1)).argmax().item()])
#     inputs+=output
#     # if output=='.': break
# print(inputs+'.')
